In [1]:
pip install psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 9.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg2
import random
import time
from datetime import datetime

# Conexión con PostgreSQL (northwind_pg = tu base fuente)
conexion = psycopg2.connect(
    host="127.0.0.1",
    port=5432,
    user="miuser",
    password="miclave",
    dbname="northwind_pg"
)
conexion.autocommit = True
cursor = conexion.cursor()

# Reutiliza supplierid y categoryid que YA existen en la tabla,
# ya que no vamos a tener cargadas las tablas Suppliers/Categories aparte.
cursor.execute("SELECT DISTINCT supplierid FROM products WHERE supplierid IS NOT NULL")
supplier_ids = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT DISTINCT categoryid FROM products WHERE categoryid IS NOT NULL")
category_ids = [row[0] for row in cursor.fetchall()]

# Calcula el siguiente productid (la tabla no es autoincremental / serial)
cursor.execute("SELECT MAX(productid) FROM products")
siguiente_id = (cursor.fetchone()[0] or 0) + 1

unidades = [
    "10 boxes x 20 bags", "24 - 12 oz bottles", "12 - 550 ml bottles",
    "1 kg pkg.", "500 g", "24 - 250 g jars", "12 - 100 g pkgs",
]

print("======================================")
print(" Generador de productos (products)")
print("======================================")
print("Insertando un producto nuevo cada 10 segundos")
print("Usa el botón de stop del kernel para detenerlo.\n")

try:
    while True:
        nombre = f"Producto automatico {siguiente_id}"
        precio = round(random.uniform(5, 200), 2)
        stock = random.randint(0, 150)
        pendiente = random.randint(0, 50)
        reorder = random.choice([0, 5, 10, 15, 25])
        descontinuado = random.random() < 0.05  # 5% de productos descontinuados

        sql = """
        INSERT INTO products
        (productid, productname, supplierid, categoryid, quantityperunit,
         unitprice, unitsinstock, unitsonorder, reorderlevel, discontinued)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        valores = (
            siguiente_id,
            nombre,
            random.choice(supplier_ids),
            random.choice(category_ids),
            random.choice(unidades),
            precio,
            stock,
            pendiente,
            reorder,
            descontinuado,
        )

        cursor.execute(sql, valores)

        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"productid: {siguiente_id} | {nombre} | "
            f"supplier: {valores[2]} | categoria: {valores[3]} | precio: ${precio:.2f}"
        )

        siguiente_id += 1
        time.sleep(10)

except KeyboardInterrupt:
    print("\nProceso detenido.")

finally:
    cursor.close()
    conexion.close()
    print("Conexión PostgreSQL cerrada.")


 Generador de productos (products)
Insertando un producto nuevo cada 10 segundos
Usa el botón de stop del kernel para detenerlo.

[15:37:05] productid: 78 | Producto automatico 78 | supplier: 28 | categoria: 4 | precio: $84.44
[15:37:15] productid: 79 | Producto automatico 79 | supplier: 28 | categoria: 8 | precio: $191.53
[15:37:25] productid: 80 | Producto automatico 80 | supplier: 8 | categoria: 3 | precio: $104.57
[15:37:35] productid: 81 | Producto automatico 81 | supplier: 21 | categoria: 8 | precio: $179.21
[15:37:45] productid: 82 | Producto automatico 82 | supplier: 21 | categoria: 4 | precio: $115.96
[15:37:55] productid: 83 | Producto automatico 83 | supplier: 29 | categoria: 6 | precio: $91.81
[15:38:05] productid: 84 | Producto automatico 84 | supplier: 7 | categoria: 3 | precio: $61.87
[15:38:15] productid: 85 | Producto automatico 85 | supplier: 13 | categoria: 5 | precio: $183.13
[15:38:25] productid: 86 | Producto automatico 86 | supplier: 6 | categoria: 3 | precio: $1